# Flet Basics

Flet enables developers to easily build **multi-platform apps** in Python (realtime web, mobile and desktop) with no frontend experience required. The basic UI elements or widgets in Flet are called **controls** which are based on [Flutter](https://flutter.dev/). Controls are designed to follow best UI practices and have sensible defaults, so applications looks good and polished by default with minimal styling effort during development.

## Hello, world!

We create a minimal app for showing random translations of "Hello, world!"™ as follows:

```bash
mkdir flet-basics && cd flet-basics
uv init --python=3.9
uv venv
uv add "flet[all]"
uv run flet create
```

This will have the following folder structure:

```bash
.
├── README.md
├── pyproject.toml
└── src
    ├── assets
    │   ├── icon.png
    │   └── splash_android.png
    ├── flet_basics
    │   └── __init__.py
    └── main.py
```

:::{.callout-note}
The original `pyproject.toml` created by `uv init` will be replaced with the one from Flet app template.
You may have to modify the values depending on your project's needs.
:::

Then, we code the main file as follows:

```{.python filename="src/main.py"}
import flet as ft
import time
from random import randint


hello_world = [
    "Hello, world!",
    "¡Hola, mundo!",
    "Bonjour, monde !",
    "Hallo, Welt!",
    "Ciao, mondo!",
    "Olá, mundo!",
    "こんにちは、世界！",
    "안녕하세요, 세계!",
    "你好，世界！",
    "مرحباً، يا عالم!",
]


def main(page: ft.Page):    # <1>
    default = hello_world[0]
    greeting = ft.Text(default, size=60, data=default)    #<2>
    n = len(hello_world)
    
    def roll_greeting(e):   # <3>
        # Force update rolling animation
        greeting.value = ""
        page.update()
        time.sleep(0.2)
        
        # Force update final result
        greeting.data = hello_world[randint(0, n - 1)]
        greeting.value = str(greeting.data)
        page.update()


    page.floating_action_button = ft.FloatingActionButton(  # <4>
        content=ft.Icon(ft.Icons.CASINO, size=60),
        on_click=roll_greeting,
        height=60, width=60
    )

    page.add(   # <5>
        ft.Container(
            greeting,
            alignment=ft.alignment.center,
            expand=True
        )
    )


if __name__ == "__main__":
    ft.app(main)
```

1. The `main` function is what Flet calls to build the page during `ft.app`. This takes a `page` variable which is updated inside the function.
2. Here we encounter our first control `Text` which we initialize with the default string `"Hello, world!"`. Note that a control distinguishes between **data** (internal to the program) and **value** (UI-facing).
3. We define a button **callback** for the event `e` (a button click) used below. This defines the behavior of the button click. Note that we explicitly update the page after showing a transitory animation state `""` after a click (so that a click is unambiguously indicated visually).
4. Assigning a floating action button with the above callback to the page. Note the use of dice icon to style the button which is available in the [Icons library](https://flet.dev/docs/reference/icons/).
5. The counter is added within a container[^container] to configure alignment. Here `expand=True` forces the container to fill the page so centering works vertically and horizontally.

[^container]: Container also implements features like padding, margin, background color, border, width & height, clipping, shape, alignment relative to the container box.

Then, run the app using `flet run` which automatically detects[^path] the `src/main` module:

[^path]: Assuming you have `path = "src"` in the `[tool.flet.app]` section of `pyproject.toml`.

<video
  src="./img/greeting.mov"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

:::{.callout-tip}
Other ways of running the app:
```bash
flet run --web app.py                 # web browser on random TCP port
flet run --web --port 8000 app.py     # web browser on port 8000
flet run -d app.py                    # hot reload current dir changes
flet run -d -r app.py                 # hot reload current dir changes + all subdirs
```
:::

## Imperative UI approach

Observe that the UI is created step-by-step with layout and behavior specified. This highlights the **imperative** style of Flet
for building UIs. To handle state changes, widget properties are directly mutated in-place, then manual calls to `page.update()` are made
to force the UI to refresh. Moreover, event handlers directly manipulate the UI. This allows arbitrary Python code to be interspersed between Flet components,
and makes it very easy to track program behavior and design for simple apps.

:::{.callout-note}
This also means that it may be difficult to create complex or large apps 
since app logic, state, and UI live in the same place.
For example, it may be tricky to react to state changes in response to, say, data changes in the backend instead of direct events like clicks.
See [Declarative UI in Flet](https://flet.dev/blog/introducing-declarative-ui-in-flet) in Flet 1.0 which introduces a declarative approach alongside the existing imperative API. 
The docs describe the declarative approach succintly as `UI = f(state)` where our code `f` describes how the UI should look like for a given state, not how to build or update it.
:::

## Flet controls

As mentioned above, the UI is made of **controls** (aka widgets) with Page as the top-most control. Controls are nested into each other and can be represented as a tree with Page as the root node. 
This picture is consistent with the imperative approach described above that we used above. Note that controls are just 
Python classes and so we can create one using their constructors:

In [7]:
import flet as ft
issubclass(ft.Text, ft.Control) 

True

### Rows

Controls are added to a page using `page.add`. Alternatively, for a control `t`, one can do `page.controls.append(t)`  followed by `page.update` to display the control in the page. We already saw how to update the UI by updating the `value` of a control and then updating the page. Some controls contain other controls:

```python
page.add(
    ft.Row(controls=[
        ft.Text("A"),
        ft.Text("B"),
        ft.Text("C")
    ])
)
```

The control `ft.Row` will arrange controls together in a row. 

### Controls list

Note that one can even dynamically remove a control from the controls list.

```python
timer = ft.Text()
page.add(timer)

while True:
    timer.data = 3
    for _ in range(3):
        timer.value = f"Removing controls in {timer.data} seconds..."
        timer.data -= 1
        page.update()
        time.sleep(1)

    page.controls[0].controls.pop()
    page.update()
    
    if len(page.controls[0].controls) == 0:
        timer.value = "No more controls!"
        page.update()
        break
```

<video
  src="./img/controls.mov"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

### Buttons and text fields

The following is the common pattern of a text field with hint and button:

In [ ]:
def main(page):
    def add_clicked(e):                                 
        page.add(ft.Checkbox(label=new_task.value))         # <1>
        new_task.value = "" # <2>
        new_task.focus()    
        new_task.update()   

    new_task = ft.TextField(hint_text="What's needs to be done?", width=300)
    submit_button = ft.ElevatedButton("Submit", on_click=add_clicked)
    page.add(ft.Row([new_task, submit_button]))

1. A submit event creates a checkbox below (page refreshes).
2. This is followed by clearing the field. Focus allows the user to not have to click the field again by having the keyboard caret in the `new_task` control -- this assumes the user typically adds multiple tasks. Finally, only the `new_task` control updates. (The page updates at the start in `page.add`.)

Note that the UI first builds with the the text field and submit button at the start. Then, the default in `ft.TextField` is cleared after clicking the text field which is sensible. Once the user clicks submit, the program updates the page and the text field: (1) page is reloaded with the new task checkbox appended, (2) text field value is set to blank and the keyboard focus us marked. Once these are set, the text field then updates.

<video
  src="./img/text-submit.mov"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

:::{.callout-note}
**Event handlers** are arbitrary functions that can modify any control in the program, as well as state variables (values and data). It can also refresh any control or the entire page.
:::

### Visible / disabled flags

Every control can be disabled using its `.disable` Boolean attribute. 
Though these are typically applied to data entry controls.
Note that the attribute can 
also be set in the constructor. Similarly, there is a `.visible` flag which
essentially prevents the controls from being rendered in the UI. Note that these 
flags are propagated recursively to every children of these controls.

```{.python filename="src/flags.py (snippets)"}
# same as above but disabled
new_task = ft.TextField(hint_text="What's needs to be done?", width=300)
submit_button = ft.ElevatedButton("Submit", on_click=add_clicked)
submit_button.disabled = True
new_task.disabled = True

# another example for modifying visibility
def make_visible_clicked(e):
    invisible_field.visible = True
    invisible_field.update()

invisible_field = ft.TextField(hint_text="Invisible field", width=300)
invisible_field.visible = False
make_visible_button = ft.ElevatedButton("???", on_click=make_visible_clicked)

page.add(ft.Row([new_task, submit_button]))
page.add(ft.Row([invisible_field, make_visible_button]))
```

:::{.callout-tip}
A useful trick is to set the flags as some conditional of state variables, e.g. `visible=(count > 0)`.
:::

Observe that the button shifts left when the text field is hidden, then shifts right:

<video
  src="./img/flags.mov"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

### Dropdown

Dropdowns are implemented by passing a list of options which are `ft.dropdown.Option` objects. 
The first element of an `Option` is the `key`. Note that the option's `key` is assigned as the 
dropdown's `value` when you select it. Moreover, the data of an option is not assigned as the
data of the dropdown. As a fun exercise, we implement a helper property for the dropdown which 
checks any of the option and gets its data whenever the key matches.

```{.python filename="src/dropdown.py"}
import flet as ft

def main(page: ft.Page):
    colored_square = ft.Container(
        width=45,
        height=45,
        bgcolor="#3B3B3B"
    )

    def button_clicked(e):
        output_text.value = f"Dropdown value is: {dropdown.value} {dropdown.selected_data}"
        colored_square.bgcolor = dropdown.selected_data
        page.update()

    output_text = ft.Text()
    submit_button = ft.ElevatedButton(text="Submit", on_click=button_clicked)

    dropdown = ft.Dropdown(
        width=150,
        options=[
            ft.dropdown.Option("Red",   data="#FF0000"),
            ft.dropdown.Option("Green", data="#00FF00"),
            ft.dropdown.Option("Blue",  data="#0000FF"),
        ],
    )

    # Attach a small helper property to the dropdown
    @property
    def selected_data(self):
        selected = next((opt for opt in self.options if opt.key == self.value), None)
        return selected.data if selected else None

    # Add the property dynamically
    dropdown.__class__.selected_data = selected_data

    page.add(
        ft.Row([dropdown, colored_square]), 
        submit_button, 
        output_text
    )

ft.app(main)
```

<video
  src="./img/dropdown.mov"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

## Custom controls

Since controls are classes, we can subclass them to get modified behavior. A common use-case is to create a reusable **restyled** control. Note that we can include other attributes of the original class (e.g. the event handler `on_click`) in the constructor that we will need later. The following sets background color to orange and text color in the button to green:

```python
class MyButton(ft.ElevatedButton):
    def __init__(self, text, on_click=None):
        super().__init__()
        self.bgcolor = ft.Colors.ORANGE_300
        self.color = ft.Colors.GREEN_800
        self.text = text
        self.on_click = on_click
```

### Composite controls

These inherit from container controls such as `Column`, `Row`, and `Stack` to combine multiple Flet controls.
The following implements a **task item** as a row control with edit button. A text field and save button appears 
when the edit button is clicked. This example also shows how to refactor a program to make it easy to read and 
make the main function simpler.

```{.python filename="src/composite.py"}
import flet as ft

class Task(ft.Row):
    def __init__(self, text):
        super().__init__()
        self.text_view      = ft.Text(text)
        self.edit_button    = ft.IconButton(icon=ft.Icons.EDIT, on_click=self.edit)
        self.edit_view      = ft.TextField(text, visible=False)
        self.save_button    = ft.IconButton(visible=False, icon=ft.Icons.SAVE, on_click=self.save)
        self.controls = [       # <1>
            ft.Checkbox(),
            self.text_view, 
            self.edit_button,   # <2>   
            self.edit_view, 
            self.save_button,   # <3>
        ]

    def edit(self, e):
        self.text_view.visible      = False
        self.edit_button.visible    = False
        self.edit_view.visible      = True
        self.save_button.visible    = True
        self.update()

    def save(self, e):
        self.text_view.visible      = True
        self.edit_button.visible    = True
        self.edit_view.visible      = False
        self.save_button.visible    = False
        self.text_view.value        = self.edit_view.value
        self.update()

    def is_isolated(self):  # <4>
        return True


def main(page: ft.Page):
    page.add(
        Task(text="Do laundry"),
        Task(text="Cook dinner"),
    )


if __name__ == "__main__":
    ft.app(main)
```

1. A task item consists of a **row** of [checkbox]{.underline}, [text]{.underline}, [edit button]{.underline}, [edit field]{.underline}, and a [save button]{.underline}.
2. Edit button hides the first two and shows the last two. This can be thought of as **edit mode.** 
3. This is reversed with save and in addition the text field receives the value 
of the edit field value. Here we are back to **view mode** (default).
4. See next section.

:::{.callout-tip}
Thinking of **state phases** of the application can be helpful (e.g. view vs. edit mode) when designing or understanding UI code.
:::

<video
  src="./img/composite.mov"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

### Isolated controls

`Task` defined above is a custom control that contains five children:

```python
Task (Row)
 ├── Checkbox
 ├── Text (view mode)
 ├── TextField (edit mode)
 ├── IconButton (edit button)
 └── IconButton (save button)
```

When edit or save button is clicked, the task: 1) hides some children, 
2) shows other children, 3), changes values, 4) then calls `self.update()`. 
During rendering when `page` updates or when an action triggers a page-level update, Flet creates a **diff** of the entire page by walking through all child components of it. Here we have two tasks. It will diff all 5 children of each task. 

Thus, when a task changes and calls `self.update()`, then `page` also calls update since something in the page changed.
Flet will try to diff the children again — even though their internal updates were already sent. This creates inefficient double diffing as discussed above. Moreover, in more complex applications we may get **patch collisions** (updating the same components at the same time causing conflicts), **UI desync** (visible inferface does not reflect true state), or artifacts like text flickering.

As such, it is recommended practice in Flet to **isolate** controls that [mutate themselves internally]{.underline} (i.e. calls `self.update()` in any of its methods). To do this, return `True` from a `.is_isolated()` call.

## Navigation and routing